## 3.1 Data Loading and Parsing

In [3]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path

DATA_DIR = Path("../MINDsmall_train")

### Load data

In [9]:
behaviour_cols = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions"
]

behaviours = pd.read_csv(
    DATA_DIR / "behaviors.tsv",
    sep="\t",
    header=None,
    names=behaviour_cols,
    dtype=str
)

news_cols = [
    "news_id", "category", "subcategory", "title", "abstract",
    "url", "title_entities", "abstract_entities"
]

news = pd.read_csv(
    DATA_DIR / "news.tsv",
    sep="\t",
    header=None,
    names=news_cols,
    dtype=str
)

print("Behaviours shape:", behaviours.shape)
display(behaviours.head())

print("News shape:", news.shape)
display(news.head())


Behaviours shape: (156965, 5)


,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


News shape: (51282, 8)


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


### Parse timestamp

In [21]:
behaviours["time_parsed"] = pd.to_datetime(
    behaviours["time"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

behaviours["time_parsed"].describe()

count                        156965
mean     2019-11-12 09:36:00.672914
min             2019-11-09 00:00:19
25%             2019-11-11 08:37:59
50%             2019-11-12 11:08:28
75%             2019-11-13 13:30:14
max             2019-11-14 23:59:13
Name: time_parsed, dtype: object

### Parse reading history

In [22]:
behaviours["history_list"] = (
    behaviours["history"]
    .fillna("")
    .apply(lambda x: x.split() if x else [])
)

behaviours["history_length"] = behaviours["history_list"].str.len()

behaviours[["history", "history_list", "history_length"]].head()

,history,history_list,history_length
0,N55189 N42782 N34694 N45794 N18445 N63302 N104...,"[N55189, N42782, N34694, N45794, N18445, N6330...",9
1,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,"[N31739, N6072, N63045, N23979, N35656, N43353...",82
2,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,"[N10732, N25792, N7563, N21087, N41087, N5445,...",16
3,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,"[N45729, N2203, N871, N53880, N41375, N43142, ...",10
4,N10078 N56514 N14904 N33740,"[N10078, N56514, N14904, N33740]",4


### Parse impressions and labels

In [23]:
def parse_impressions(s):
    result = []

    for token in s.split():
        news_id, label = token.rsplit("-", 1)

        result.append({
            "news_id": news_id,
            "label": int(label)
        })

    return result


behaviours["parsed_impressions"] = behaviours["impressions"].apply(parse_impressions)

behaviours["num_candidates"] = behaviours["parsed_impressions"].str.len()

behaviours["num_clicked"] = behaviours["parsed_impressions"].apply(lambda x: sum(item["label"] for item in x))

behaviours["num_unclicked"] = (behaviours["num_candidates"] - behaviours["num_clicked"])

display(
    behaviours[
        [
            "impression_id",
            "user_id",
            "num_candidates",
            "num_clicked",
            "num_unclicked"
        ]
    ].head()
)

,impression_id,user_id,num_candidates,num_clicked,num_unclicked
0,1,U13740,2,1,1
1,2,U91836,11,1,10
2,3,U73700,36,1,35
3,4,U34670,4,1,3
4,5,U8125,69,1,68


### Load embedding files

In [10]:
def load_embedding(path, id_col):
    df = pd.read_csv(
        path,
        sep="\t",
        header=None
    )

    # Remove completely empty columns, e.g. caused by trailing tabs
    df = df.dropna(axis=1, how="all")

    embedding_dim = df.shape[1] - 1

    df.columns = (
        [id_col]
        + [f"emb_{i}" for i in range(embedding_dim)]
    )

    # Ensure embeddings are numeric
    embedding_cols = [c for c in df.columns if c != id_col]
    df[embedding_cols] = df[embedding_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    return df

In [11]:
entity_embeddings = load_embedding(
    DATA_DIR / "entity_embedding.vec",
    "entity_id"
)

relation_embeddings = load_embedding(
    DATA_DIR / "relation_embedding.vec",
    "relation_id"
)

print("Entity embeddings shape:", entity_embeddings.shape)
print("Relation embeddings shape:", relation_embeddings.shape)

display(entity_embeddings.head())

Entity embeddings shape: (26904, 101)
Relation embeddings shape: (1091, 101)


,entity_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_90,emb_91,emb_92,emb_93,emb_94,emb_95,emb_96,emb_97,emb_98,emb_99
0,Q41,-0.063388,-0.181451,0.057501,-0.091254,-0.076217,-0.052525,0.050500,-0.224871,-0.018145,...,-0.051949,0.001861,0.124535,-0.151043,-0.263698,-0.103607,0.020007,-0.101157,-0.091567,0.035234
1,Q1860,0.060958,0.069934,0.015832,0.079471,-0.023362,-0.125007,-0.043618,0.134063,-0.121691,...,-0.070713,-0.014287,0.013578,0.099977,0.012199,-0.141138,0.056129,-0.133727,0.025795,0.051448
2,Q39631,-0.093106,-0.052002,0.020556,-0.020801,0.043180,-0.072321,0.000910,0.028156,0.176303,...,-0.124472,-0.086840,-0.078992,-0.062712,0.051117,-0.184307,0.127637,-0.144866,0.044690,0.013498
3,Q30,-0.115737,-0.179113,0.102739,-0.112469,-0.101853,-0.177516,0.015860,-0.092626,0.086708,...,0.005893,0.080511,-0.000085,-0.089968,-0.083486,-0.149992,-0.053031,-0.136071,-0.029001,0.174155
4,Q60,-0.051036,-0.165637,0.132802,-0.089949,-0.146637,-0.142246,0.103853,-0.129651,0.096265,...,-0.002713,0.078628,0.003711,-0.058953,-0.154067,-0.117159,-0.031614,-0.140451,0.001288,0.140350


### Parse news entities

In [12]:
def parse_entities(x):
    if pd.isna(x) or x == "":
        return []

    try:
        parsed = json.loads(x)

        if isinstance(parsed, list):
            return parsed

        return None

    except (json.JSONDecodeError, TypeError):
        return None

In [13]:
news["title_entities_parsed"] = (
    news["title_entities"].apply(parse_entities)
)

news["abstract_entities_parsed"] = (
    news["abstract_entities"].apply(parse_entities)
)

### Derived features

In [14]:
news["num_title_entities"] = (
    news["title_entities_parsed"]
    .apply(lambda x: len(x) if isinstance(x, list) else np.nan)
)

news["num_abstract_entities"] = (
    news["abstract_entities_parsed"]
    .apply(lambda x: len(x) if isinstance(x, list) else np.nan)
)

news["title_length"] = (
    news["title"]
    .fillna("")
    .str.split()
    .str.len()
)

news["abstract_length"] = (
    news["abstract"]
    .fillna("")
    .str.split()
    .str.len()
)

## Data Quality and Missing Values

In [16]:
def missing_summary(df):
    summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100
    })

    return summary.sort_values(
        "missing_pct",
        ascending=False
    )

In [17]:
print("Behaviors missing values")
display(missing_summary(
    behaviours[behaviour_cols]
))

print("News missing values")
display(missing_summary(
    news[news_cols]
))

Behaviors missing values


,missing_count,missing_pct
history,3238,2.06288
impression_id,0,0.00000
user_id,0,0.00000
time,0,0.00000
impressions,0,0.00000


News missing values


,missing_count,missing_pct
abstract,2666,5.198705
abstract_entities,4,0.007800
title_entities,3,0.005850
news_id,0,0.000000
category,0,0.000000
subcategory,0,0.000000
title,0,0.000000
url,0,0.000000


### Duplicates

In [18]:
print("Duplicate behavior rows:",
      behaviours.duplicated(subset=behaviour_cols).sum())

print("Duplicate impression IDs:",
      behaviours["impression_id"].duplicated().sum())

print("Duplicate news rows:",
      news.duplicated(subset=news_cols).sum())

print("Duplicate news IDs:",
      news["news_id"].duplicated().sum())

print("Duplicate entity IDs:",
      entity_embeddings["entity_id"].duplicated().sum())

print("Duplicate relation IDs:",
      relation_embeddings["relation_id"].duplicated().sum())

Duplicate behavior rows: 0
Duplicate impression IDs: 0
Duplicate news rows: 0
Duplicate news IDs: 0
Duplicate entity IDs: 0
Duplicate relation IDs: 0


### Identifier validity

In [19]:
invalid_users = (
    ~behaviours["user_id"]
    .fillna("")
    .str.match(r"^U\d+$")
)

invalid_news = (
    ~news["news_id"]
    .fillna("")
    .str.match(r"^N\d+$")
)

print("Invalid user IDs:", invalid_users.sum())
print("Invalid news IDs:", invalid_news.sum())

Invalid user IDs: 0
Invalid news IDs: 0


### Timestamp validity

In [24]:
print(
    "Unparseable timestamps:",
    behaviours["time_parsed"].isna().sum()
)

print(
    "Timestamp range:",
    behaviours["time_parsed"].min(),
    "to",
    behaviours["time_parsed"].max()
)

Unparseable timestamps: 0
Timestamp range: 2019-11-09 00:00:19 to 2019-11-14 23:59:13


### Impression-format validation

In [25]:
impression_pattern = re.compile(r"^N\d+-[01]$")

invalid_impression_tokens = []

for impression_string in behaviours["impressions"].dropna():

    for token in impression_string.split():

        if not impression_pattern.match(token):
            invalid_impression_tokens.append(token)

print(
    "Invalid impression tokens:",
    len(invalid_impression_tokens)
)

Invalid impression tokens: 0


### Click-class imbalance

In [26]:
total_clicked = behaviours["num_clicked"].sum()
total_unclicked = behaviours["num_unclicked"].sum()

total_candidates = (
    total_clicked + total_unclicked
)

click_rate = (
    total_clicked / total_candidates
)

print("Clicked candidates:", total_clicked)
print("Unclicked candidates:", total_unclicked)
print("Total candidates:", total_candidates)
print(f"Click rate: {click_rate:.2%}")

Clicked candidates: 236344
Unclicked candidates: 5607100
Total candidates: 5843444
Click rate: 4.04%


### History & candidate distribution analysis

In [27]:
display(
    behaviours[
        [
            "history_length",
            "num_candidates",
            "num_clicked"
        ]
    ].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

,history_length,num_candidates,num_clicked
count,156965.000000,156965.000000,156965.000000
mean,32.539987,37.227688,1.505711
std,40.267607,38.594148,1.153102
min,0.000000,2.000000,1.000000
25%,8.000000,10.000000,1.000000
50%,19.000000,24.000000,1.000000
75%,42.000000,51.000000,2.000000
90%,78.000000,91.000000,3.000000
95%,109.000000,118.000000,4.000000
99%,190.000000,166.000000,6.000000


### News category distributions

In [28]:
print("Number of categories:",
      news["category"].nunique())

print("Number of subcategories:",
      news["subcategory"].nunique())

display(
    news["category"]
    .value_counts()
    .to_frame("count")
)

display(
    news["subcategory"]
    .value_counts()
    .head(20)
    .to_frame("count")
)

Number of categories: 17
Number of subcategories: 264


,count
category,
news,15774
sports,14510
finance,3107
foodanddrink,2551
lifestyle,2479
travel,2350
video,2068
weather,2048
health,1885


,count
subcategory,
newsus,6564
football_nfl,5420
newspolitics,2826
newscrime,2254
weathertopstories,2047
newsworld,1720
football_ncaa,1665
baseball_mlb,1661
basketball_nba,1555


### News text quality

In [29]:
display(
    news[
        [
            "title_length",
            "abstract_length",
            "num_title_entities",
            "num_abstract_entities"
        ]
    ].describe()
)

,title_length,abstract_length,num_title_entities,num_abstract_entities
count,51282.000000,51282.000000,51282.000000,51282.000000
mean,10.754417,34.293319,1.168987,1.858527
std,3.265311,26.542819,0.977183,1.822684
min,1.000000,0.000000,0.000000,0.000000
25%,9.000000,15.000000,0.000000,0.000000
50%,10.000000,24.000000,1.000000,1.000000
75%,13.000000,62.000000,2.000000,3.000000
max,57.000000,474.000000,8.000000,30.000000


In [30]:
print(
    "Empty/missing titles:",
    news["title"].fillna("").str.strip().eq("").sum()
)

print(
    "Empty/missing abstracts:",
    news["abstract"].fillna("").str.strip().eq("").sum()
)

Empty/missing titles: 0
Empty/missing abstracts: 2666


### Check malformed entity JSON

In [31]:
invalid_title_json = (
    news["title_entities_parsed"]
    .apply(lambda x: x is None)
    .sum()
)

invalid_abstract_json = (
    news["abstract_entities_parsed"]
    .apply(lambda x: x is None)
    .sum()
)

print(
    "Malformed title entity JSON:",
    invalid_title_json
)

print(
    "Malformed abstract entity JSON:",
    invalid_abstract_json
)

Malformed title entity JSON: 0
Malformed abstract entity JSON: 0


### Referential integrity betewen behaviours and news

In [32]:
behaviour_news_ids = set()

# IDs appearing in user histories
for history in behaviours["history"].dropna():
    behaviour_news_ids.update(history.split())

# IDs appearing in candidate impressions
for impressions in behaviours["impressions"].dropna():

    for token in impressions.split():
        news_id = token.rsplit("-", 1)[0]
        behaviour_news_ids.add(news_id)

news_ids = set(news["news_id"])

missing_from_news = (
    behaviour_news_ids - news_ids
)

print(
    "Unique news IDs referenced in behaviours:",
    len(behaviour_news_ids)
)

print(
    "Referenced IDs absent from news.tsv:",
    len(missing_from_news)
)

Unique news IDs referenced in behaviours: 51282
Referenced IDs absent from news.tsv: 0


### Check entity-embedding coverage

In [33]:
def extract_wikidata_ids(entity_list):

    if not isinstance(entity_list, list):
        return []

    return [
        entity["WikidataId"]
        for entity in entity_list
        if "WikidataId" in entity
    ]

In [34]:
news["title_wikidata_ids"] = (
    news["title_entities_parsed"]
    .apply(extract_wikidata_ids)
)

news["abstract_wikidata_ids"] = (
    news["abstract_entities_parsed"]
    .apply(extract_wikidata_ids)
)

In [35]:
news_entity_ids = set()

for ids in news["title_wikidata_ids"]:
    news_entity_ids.update(ids)

for ids in news["abstract_wikidata_ids"]:
    news_entity_ids.update(ids)

embedding_entity_ids = set(
    entity_embeddings["entity_id"]
)

entities_without_embedding = (
    news_entity_ids - embedding_entity_ids
)

print(
    "Unique entities referenced in news:",
    len(news_entity_ids)
)

print(
    "Entities with embeddings:",
    len(news_entity_ids & embedding_entity_ids)
)

print(
    "Entities without embeddings:",
    len(entities_without_embedding)
)

if len(news_entity_ids) > 0:
    print(
        "Entity embedding coverage:",
        f"{len(news_entity_ids & embedding_entity_ids) / len(news_entity_ids):.2%}"
    )

Unique entities referenced in news: 27759
Entities with embeddings: 26902
Entities without embeddings: 857
Entity embedding coverage: 96.91%


### Embedding quality

In [36]:
def check_embedding_quality(df, id_col):

    embedding_cols = [
        col for col in df.columns
        if col != id_col
    ]

    values = df[embedding_cols].to_numpy()

    return pd.Series({
        "rows": len(df),
        "dimensions": len(embedding_cols),
        "duplicate_ids": df[id_col].duplicated().sum(),
        "missing_values":
            df[embedding_cols].isna().sum().sum(),
        "non_finite_values":
            (~np.isfinite(values)).sum()
    })

In [37]:
embedding_quality = pd.DataFrame({
    "entity_embedding": check_embedding_quality(
        entity_embeddings,
        "entity_id"
    ),
    "relation_embedding": check_embedding_quality(
        relation_embeddings,
        "relation_id"
    )
}).T

display(embedding_quality)

,rows,dimensions,duplicate_ids,missing_values,non_finite_values
entity_embedding,26904,100,0,0,0
relation_embedding,1091,100,0,0,0


## Summary

In [39]:
dataset_summary = pd.DataFrame({
    "Dataset": [
        "Behaviors",
        "News",
        "Entity embeddings",
        "Relation embeddings"
    ],
    "Rows": [
        len(behaviours),
        len(news),
        len(entity_embeddings),
        len(relation_embeddings)
    ],
    "Columns": [
        len(behaviour_cols),
        len(news_cols),
        entity_embeddings.shape[1],
        relation_embeddings.shape[1]
    ]
})

display(dataset_summary)

,Dataset,Rows,Columns
0,Behaviors,156965,5
1,News,51282,8
2,Entity embeddings,26904,101
3,Relation embeddings,1091,101
